# Análisis de Competencia Bancaria

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Analizar posicionamiento competitivo** con `CORTEX.COMPLETE()`
2. **Comparar productos y pricing** entre entidades
3. **Extraer inteligencia competitiva** de textos públicos
4. **Generar briefings** de competencia automatizados
5. **Construir dashboard** de benchmarking

---

## Lo Que Construirás

Un **sistema de inteligencia competitiva** bancaria:
- Comparativa de productos, precios y servicios
- Análisis de sentimiento de competidores en medios
- Briefings ejecutivos automáticos
- Dashboard de benchmarking

**Valor de Negocio**: Anticipar movimientos de la competencia y ajustar estrategia.

---

## Funcionalidades Clave

- `CORTEX.COMPLETE()` — Análisis y comparativas
- `CORTEX.SENTIMENT()` — Sentimiento sobre competidores
- `GENERATOR()` — Datos de mercado sintéticos

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS COMPETENCIA_DB;
CREATE SCHEMA IF NOT EXISTS COMPETENCIA_DB.PUBLIC;
USE SCHEMA COMPETENCIA_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Datos de Competencia

### Qué Vamos a Crear

- **8 bancos competidores** con productos y precios
- **500 noticias/menciones** sobre competidores
- **Comparativas de productos** (hipotecas, fondos, tarjetas)

In [ ]:
-- Bancos competidores
CREATE OR REPLACE TABLE COMPETIDORES (
    banco VARCHAR(30) PRIMARY KEY,
    tipo VARCHAR(20),
    cuota_mercado FLOAT,
    num_oficinas INTEGER,
    satisfaccion_cliente FLOAT
);

INSERT INTO COMPETIDORES VALUES
('Santander','Tradicional',15.2,3200,7.1),
('BBVA','Tradicional',13.8,2800,7.4),
('CaixaBank','Tradicional',16.5,4000,7.0),
('Sabadell','Tradicional',5.8,1400,6.9),
('Bankinter','Tradicional',4.2,400,7.8),
('ING','Digital',3.5,0,8.1),
('Openbank','Digital',2.1,0,7.6),
('Revolut','Neobank',1.8,0,7.9);

-- Productos competidores
CREATE OR REPLACE TABLE PRODUCTOS_COMPETENCIA (
    banco VARCHAR(30),
    producto VARCHAR(30),
    tipo_interes DECIMAL(5,3),
    comision_mensual DECIMAL(5,2),
    ventaja_diferencial TEXT,
    PRIMARY KEY (banco, producto)
);

INSERT INTO PRODUCTOS_COMPETENCIA VALUES
('Santander','Hipoteca Fija',2.990,0,'Tipo fijo 30 años'),
('BBVA','Hipoteca Variable',1.490,0,'Euribor + 0.99%'),
('CaixaBank','Hipoteca Mixta',2.490,0,'5 años fijo + variable'),
('ING','Hipoteca Sin Comisiones',2.790,0,'Cero comisiones apertura'),
('Bankinter','Cuenta Nómina',0,0,'5% TAE primer año'),
('Revolut','Cuenta Multi-divisa',0,0,'Cambio FX sin comisión'),
('Openbank','Fondo Indexado',0.30,0,'TER 0.30%'),
('BBVA','Tarjeta Aqua',0,3.00,'Sin números impresos');

-- Noticias de competidores
CREATE OR REPLACE TABLE NOTICIAS_COMPETENCIA (
    noticia_id VARCHAR(10) PRIMARY KEY,
    banco VARCHAR(30),
    fecha DATE,
    titular TEXT,
    fuente VARCHAR(30)
);

INSERT INTO NOTICIAS_COMPETENCIA
SELECT
    'NOT' || LPAD(SEQ4()::STRING, 5, '0'),
    ARRAY_CONSTRUCT('Santander','BBVA','CaixaBank','Sabadell','Bankinter','ING','Openbank','Revolut')[UNIFORM(0,7,RANDOM())]::VARCHAR,
    DATEADD('day', -UNIFORM(0, 90, RANDOM()), CURRENT_DATE()),
    CASE UNIFORM(1,8,RANDOM())
        WHEN 1 THEN 'El banco lanza nueva app con funcionalidades de IA para asesoramiento personalizado'
        WHEN 2 THEN 'Resultados trimestrales superan expectativas con un 12% de crecimiento en comisiones'
        WHEN 3 THEN 'Multado con 5 millones por deficiencias en prevención de blanqueo de capitales'
        WHEN 4 THEN 'Anuncia el cierre de 200 oficinas en su plan de transformación digital'
        WHEN 5 THEN 'Lanza hipoteca verde con tipo preferencial para viviendas eficientes'
        WHEN 6 THEN 'Sufre caída del servicio durante 6 horas afectando a millones de clientes'
        WHEN 7 THEN 'Acuerdo estratégico con fintech para pagos instantáneos en Europa'
        ELSE 'El CEO anuncia plan de inversión de 500M en tecnología para los próximos 3 años'
    END,
    ARRAY_CONSTRUCT('Expansión','Cinco Días','El Economista','Bloomberg','Reuters')[UNIFORM(0,4,RANDOM())]::VARCHAR
FROM TABLE(GENERATOR(ROWCOUNT => 500));

SELECT banco, COUNT(*) AS noticias FROM NOTICIAS_COMPETENCIA GROUP BY 1 ORDER BY 2 DESC;

---

## Paso 3: Análisis de Sentimiento Competitivo

Analizamos el sentimiento de las noticias de cada competidor.

In [ ]:
CREATE OR REPLACE TABLE SENTIMIENTO_COMPETENCIA AS
SELECT
    noticia_id, banco, fecha, titular, fuente,
    SNOWFLAKE.CORTEX.SENTIMENT(titular) AS score,
    CASE
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(titular) < -0.3 THEN 'Negativo'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(titular) > 0.3 THEN 'Positivo'
        ELSE 'Neutro'
    END AS sentimiento
FROM NOTICIAS_COMPETENCIA;

SELECT banco, sentimiento, COUNT(*), ROUND(AVG(score),3) AS score_medio
FROM SENTIMIENTO_COMPETENCIA GROUP BY 1, 2 ORDER BY 1, 2;

---

## Paso 4: Briefing Competitivo con IA

Generamos un briefing ejecutivo comparando nuestro posicionamiento.

In [ ]:
SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
    'Genera un briefing competitivo para el comité de dirección de un banco español con estos datos:\n\n' ||
    'Competidores principales: Santander (15.2% cuota), CaixaBank (16.5%), BBVA (13.8%)\n' ||
    'Digitales: ING (3.5%), Revolut (1.8%)\n' ||
    'Satisfacción cliente: ING 8.1, Bankinter 7.8, Revolut 7.9, BBVA 7.4, Santander 7.1\n' ||
    'Noticias recientes negativas: multas AML, cierre oficinas\n' ||
    'Noticias positivas: IA, hipotecas verdes, acuerdos fintech\n\n' ||
    'Genera 4 secciones: Panorama competitivo, Amenazas, Oportunidades, Recomendaciones estratégicas. En español.'
) AS briefing_competitivo;

---

## Paso 5: Dashboard de Benchmarking

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Análisis de Competencia Bancaria')
st.markdown('### Benchmarking Competitivo')

df_comp = session.sql('SELECT * FROM COMPETIDORES ORDER BY cuota_mercado DESC').to_pandas()
st.subheader('Cuota de Mercado')
st.bar_chart(df_comp.set_index('BANCO')['CUOTA_MERCADO'])

st.markdown('---')
st.subheader('Satisfacción del Cliente')
st.bar_chart(df_comp.set_index('BANCO')['SATISFACCION_CLIENTE'])

st.markdown('---')
st.subheader('Sentimiento en Medios por Banco')
df_sent = session.sql('SELECT banco, ROUND(AVG(score),3) AS score_medio FROM SENTIMIENTO_COMPETENCIA GROUP BY 1 ORDER BY 2').to_pandas()
st.bar_chart(df_sent.set_index('BANCO'))

st.markdown('---')
st.subheader('Productos Competidores')
df_prod = session.sql('SELECT * FROM PRODUCTOS_COMPETENCIA').to_pandas()
st.dataframe(df_prod)

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS COMPETENCIA_DB;